# P7 lab: build a grounded prompt, budget it, order it, check it

Lesson: [`concepts/P7_grounded_prompting.md`](../concepts/P7_grounded_prompting.md), Part H.1. Concept card: [02 P7](../02_RAG_MASTER_LEARNING_PLAN.md#p7).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. The reference script `sources/verification/P7/p7_measure.py` is a solution: open it only after an honest attempt plus a hint from your tutor.
- **Predict first.** Some tasks ask for a prediction in a cell before you compute. Fill it in honestly; it is part of the evidence.
- Tasks 1–6 make **no API calls**. Task 7 is optional and calls Groq; it is switched off by default.

## Setup (given)
Run this cell. It loads the climate PDF, defines a token counter (`n_tok`, the `o200k_harmony` encoding that tiktoken uses for `gpt-oss-20b`, as in P1) and a fingerprint `_fp` that the CHECK cells use, so expected answers stay hidden until you have predicted.

`CHUNKS4` is a **hand-picked toy "ranked list"** for question D-001. It is **not** a retrieval result. Rank 1 holds the answer; ranks 2 and 3 hold other temperature numbers (distractors).

In [ ]:
import hashlib, json, re
import pymupdf, tiktoken

PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"
ENC = tiktoken.get_encoding("o200k_harmony")

def n_tok(text):
    """Number of o200k_harmony tokens in a string."""
    return len(ENC.encode(text, disallowed_special=()))

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

pages = [p.get_text() for p in pymupdf.open(PDF)]

def _slices(spec):
    return [{"doc_id": "climate", "page": p, "text": pages[p - 1][a:b]} for p, a, b in spec]

CHUNKS4 = _slices([(3, 800, 1800), (10, 0, 1000), (23, 1200, 2200), (1, 1000, 2000)])        # toy ranks 1-4
CHUNKS8 = CHUNKS4 + _slices([(6, 1389, 2389), (24, 0, 1000), (2, 0, 1000), (8, 0, 1000)])    # toy ranks 5-8
CHUNKS4_2K = _slices([(3, 183, 2183), (10, 0, 2000), (23, 204, 2204), (1, 424, 2424)])       # same 4, 2,000 chars each

Q_D001 = ("According to the document, by how much have global temperatures risen "
          "since the late 19th century?")
Q_D009 = "In what year was the IPCC established?"          # unanswerable from this PDF (06 §6.7)

print(len(pages), "pages;", [(c["page"], len(c["text"])) for c in CHUNKS8])

## Task 1: index time or query time? (MC-003)

For each step below, write `"index"` (done once per document, before any question) or `"query"` (done for each question).

Tip: ask yourself, "Does this run once per document, or once per question?"

In [ ]:
my_stages = {
    "set chunk_size and overlap": None,
    "embed every chunk": None,
    "caption the images (captioning notebook cell 12)": None,
    "embed the question": None,
    "choose k": None,
    "order the retrieved chunks": None,
    "number the context blocks with (doc:page) headers": None,
    "insert the answer-only-from-sources rule": None,
}

In [ ]:
# CHECK Task 1 (do not edit)
assert all(v in ("index", "query") for v in my_stages.values()), "every value must be 'index' or 'query'"
assert len(my_stages) == 8, "do not add or remove steps"
assert _fp(my_stages) == "05578baa00e3bdcf", \
    "at least one step is in the wrong stage. For each one ask: once per document, or once per question?"
print("Task 1 OK")

**In your own words (one sentence).** Where does chunk overlap live, and where does the prompt live?

*Template:* "Overlap is set at ____ time, when ____. The prompt is built at ____ time, when ____."

*Your answer:*

## Task 2a: write the grounding rules from memory

Close the lesson. In `MY_RULES`, write the system rules of a grounded prompt in your own words. A good set has four ideas:
answer only from the sources; cite every fact with a `[doc:page]`-style ID; say a fixed "cannot answer" sentence when the sources are not enough; do not use outside knowledge.

The CHECK only tests that the four ideas are present. Your tutor grades the wording.

In [ ]:
MY_RULES = """
"""

In [ ]:
# CHECK Task 2a (do not edit)
_r = MY_RULES
assert re.search(r"\bonly\b", _r, re.I), "say that the answer must come ONLY from the sources"
assert re.search(r"\[[A-Za-z_]+:[A-Za-z0-9_]+\]", _r), "show the citation format, e.g. [doc_id:page] or [climate:10]"
assert re.search(r"(cannot|can't|can not|don't|do not) (answer|know)", _r, re.I), "give the sentence to say when the sources are not enough"
assert re.search(r"(outside|prior|own|external|general|background|training) (knowledge|information)|memory", _r, re.I), \
    "forbid outside knowledge"
print("Task 2a OK:", n_tok(_r), "tokens of rules")

## Task 2b: build the v0 prompt

Now use the **official** v0 rules (given below, from 06 §3.3). Write `build_prompt(question, chunks)` that returns a tuple `(system, user)`:
- `system` is exactly `SYSTEM_V0`;
- `user` starts with `Sources:` and a newline; then one block per chunk, **in the given order**, numbered from 1. A block is a header line `[i] (doc_id:page)`, a newline, then the chunk text. Blocks are separated by one empty line;
- after the last block: one empty line, then `Question: <question>`, a newline, and `Answer:` (nothing after it).

Then set `tokens_k4 = n_tok(system) + n_tok(user)` for `build_prompt(Q_D001, CHUNKS4)`.

**Predict first.** P1 measured 976 tokens for four 1,000-character windows. These are four different 1,000-character slices. Predict `tokens_k4`.

In [ ]:
SYSTEM_V0 = (  # 06 §3.3 prompts/answer_v1.txt, SYSTEM part (given)
    "You answer questions using ONLY the numbered sources below.\n"
    "Rules:\n"
    "1. Every factual sentence must end with one or more citations in the form [doc_id:page],\n"
    "   copied exactly from the source headers, e.g. [climate:10].\n"
    "2. If the sources do not contain the answer, reply exactly:\n"
    "   \"I cannot answer this from the provided documents.\"\n"
    "   Do not use outside knowledge.\n"
    "3. Do not cite a source that does not support the sentence.\n"
    "4. Be concise: at most 5 sentences.\n")
ABSTAIN = "I cannot answer this from the provided documents."
my_prediction_tokens_k4 = None    # replace None with your number, BEFORE you build the prompt

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p7_measure.py).

In [ ]:
# CHECK Task 2b (do not edit)
assert my_prediction_tokens_k4 is not None, "write your prediction first"
_s, _u = build_prompt(Q_D001, CHUNKS4)
assert _s == SYSTEM_V0, "system must be exactly SYSTEM_V0"
assert _u.startswith("Sources:\n"), "user must start with 'Sources:' and a newline"
_h = re.findall(r"^\[(\d+)\] \(([^:()]+):(\d+)\)$", _u, re.M)
assert _h == [("1", "climate", "3"), ("2", "climate", "10"), ("3", "climate", "23"), ("4", "climate", "1")], \
    f"headers must be '[i] (doc_id:page)' on their own line, numbered 1..4 in the given order; found {_h}"
assert all(c["text"] in _u for c in CHUNKS4), "every chunk text must appear verbatim"
assert _u.rfind("[4] (climate:1)") < _u.find("Question: " + Q_D001), "the question comes after the last block"
assert _u.endswith("Answer:"), "user must end with 'Answer:'"
assert _fp([_s, _u]) == "6fa99768a3dfeca8", \
    "structure is right but a separator differs: check the blank lines between blocks and before 'Question:'"
assert _fp(tokens_k4) == "debc96817a3523d6", f"tokens_k4 = {tokens_k4}: count n_tok(system) + n_tok(user)"
print(f"Task 2b OK. tokens_k4 = {tokens_k4}; you predicted {my_prediction_tokens_k4}")
print(_u[:400])

## Task 3: the budget and the answer reserve (CR-08)

Write two functions:
- `input_budget(window, max_tokens, margin)`: tokens left for the **input** after reserving `max_tokens` for the answer (reasoning included) and a safety `margin` for template overhead;
- `fit_blocks(fixed_tokens, block_tokens, budget)`: walk `block_tokens` in order and return **how many** blocks fit, so that `fixed_tokens` + the kept blocks stay `<= budget`. Stop at the first block that does not fit.

Then measure two "what if it doubles" cases:
- `tokens_k8 = n_tok(system) + n_tok(user)` for `build_prompt(Q_D001, CHUNKS8)` (k doubles from 4 to 8);
- `tokens_2k` for `build_prompt(Q_D001, CHUNKS4_2K)` (chunk size doubles from 1,000 to 2,000 characters).

**Predict first:** does each one double `tokens_k4`? Write your two numbers.

In [ ]:
my_prediction_tokens_k8 = None    # BEFORE measuring
my_prediction_tokens_2k = None    # BEFORE measuring

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p7_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_tokens_k8 is not None and my_prediction_tokens_2k is not None, "write both predictions first"
assert input_budget(8192, 512, 128) == 7552, "input_budget = window - max_tokens - margin"
assert fit_blocks(100, [200, 150, 300], 600) == 2, "100 + 200 + 150 = 450 fits; + 300 = 750 does not"
assert fit_blocks(0, [50], 40) == 0, "a block larger than the budget does not fit"
assert fit_blocks(10, [5, 5, 5], 20) == 2, "stay <= budget (20 is allowed, 25 is not)"
assert fit_blocks(0, [], 100) == 0
assert _fp(tokens_k8) == "0c75ccaac2812081", f"tokens_k8 = {tokens_k8} is not right yet"
assert _fp(tokens_2k) == "35ea9e03d5f2f465", f"tokens_2k = {tokens_2k} is not right yet"
print(f"k 4 -> 8:       {tokens_k4} -> {tokens_k8} tokens (x{tokens_k8 / tokens_k4:.2f}); you predicted {my_prediction_tokens_k8}")
print(f"size 1k -> 2k:  {tokens_k4} -> {tokens_2k} tokens (x{tokens_2k / tokens_k4:.2f}); you predicted {my_prediction_tokens_2k}")
print("Task 3 OK")

**Why is neither ratio 2.0?** One or two sentences.

*Template:* "It is less than double because ____ does not grow with k, and ____."

*Your answer:*

## Task 4: put the best evidence at the edges ("lost in the middle")

Liu et al. (2023) found that models use evidence at the **start or end** of a long context more reliably than evidence in the **middle**. One response: write `edges_first(ranked)` that takes a list in rank order (best first) and returns a new order with **rank 1 first, rank 2 last, rank 3 second, rank 4 second-to-last**, and so on, so the weakest ranks end up in the middle.

**Predict first:** for `k = 8`, which two ranks end up in the two middle positions?

In [ ]:
my_prediction_middle_ranks_k8 = None    # e.g. [?, ?], BEFORE you write the function

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p7_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_middle_ranks_k8 is not None, "write your prediction first"
_o = edges_first(list(range(1, 9)))
assert sorted(_o) == list(range(1, 9)), "edges_first must return the same items, reordered"
assert _o[0] == 1 and _o[-1] == 2, "rank 1 goes first and rank 2 goes last"
assert edges_first([]) == [] and edges_first(["a"]) == ["a"]
assert _fp(_o) == "6b966d3b7c398acc", f"order {_o} is not right yet: rank 3 second, rank 4 second-to-last, ..."
_s2, _u2 = build_prompt(Q_D001, edges_first(CHUNKS4))
print("k=8 order:", _o, "| middle:", _o[3:5], "| you predicted", my_prediction_middle_ranks_k8)
print("reordered headers:", re.findall(r"^\[\d+\] \(.*\)$", _u2, re.M))
print("Task 4 OK")

**Look at the reordered headers.** The block numbers `[1]..[4]` changed, but the `(doc:page)` labels did not. Why does v0 ask the model to cite `[doc_id:page]` and not the block number?

*Template:* "Because block numbers ____ when we ____, but `[doc:page]` ____."

*Your answer:*

## Task 5: check an answer without a judge

Write three functions:
- `extract_citations(answer)`: a list of `(doc_id, page)` tuples, `page` as an `int`, for every `[doc_id:page]` in the answer, in order;
- `invalid_citations(answer, chunks)`: the citations whose `(doc_id, page)` is **not** among the chunks that were in the prompt;
- `is_abstention(answer)`: `True` when the answer is the fixed `ABSTAIN` sentence, ignoring case, surrounding spaces and quotes, and a final full stop. Any other wording is `False`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p7_measure.py).

In [ ]:
# CHECK Task 5 (do not edit). The answers below are toy strings written for P7, not model outputs.
A = "Global temperatures have risen by about 1.2 degrees Celsius since the late 19th century [climate:3]."
B = "Temperatures rose about 1.2 degrees Celsius [climate:3]; the Paris goal is well below 2 degrees [climate:12]."
assert extract_citations(A) == [("climate", 3)], "return (doc_id, page) tuples, page as an int"
assert extract_citations(B) == [("climate", 3), ("climate", 12)]
assert invalid_citations(A, CHUNKS4) == []
assert invalid_citations(B, CHUNKS4) == [("climate", 12)], "page 12 was not in the prompt"
assert extract_citations("No citation here.") == []
assert is_abstention(ABSTAIN) is True
assert is_abstention('  "i cannot answer this from the provided documents"  ') is True
assert is_abstention("I can't answer that from these sources.") is False, "only the fixed sentence counts"
assert is_abstention("Temperatures have risen by about 1.2 degrees Celsius.") is False
print("Task 5 OK")

## Task 6: audit five repo prompts

The instruction texts below are copied from the pinned repo (cells are 0-based). Judge **only by what the cited cells show**. For each prompt, fill five booleans:
- `context_only`: the instruction says to answer only from the given context, and nothing invites the model's own knowledge;
- `refusal`: the instruction says what to do when the answer is not in the context;
- `cite_ids`: the instruction asks the model to cite source IDs;
- `labelled_blocks`: the code in the cited cells puts a source label (page, source or ID) on each block the model sees;
- `invites_memory`: the instruction invites the model to use its own knowledge.

In [ ]:
PROMPTS = {  # instruction text (given)
    "simple_csv_rag 16": ("You are an assistant for question-answering tasks. Use the following pieces of retrieved context "
                          "to answer the question. If you don't know the answer, say that you don't know. Use three "
                          "sentences maximum and keep the answer concise."),
    "local_rag_hf_faiss 22+24": ("You are a helpful assistant. Answer the question using ONLY the context provided below.\n"
                                 "If the answer is not contained in the context, respond with:\n"
                                 "\"I don't have enough information in the provided context to answer this question.\"\n"
                                 "Do not use any prior knowledge or make assumptions."),
    "helper_functions L162-205": "For the question below, provide a concise but suffice answer based ONLY on the provided context:",
    "reliable_rag 18": ("You are an assistant for question-answering tasks. Answer the question based upon your knowledge. \n"
                        "Use three-to-five sentences maximum and keep the answer concise."),
    "captioning 20": ("You are an assistant for question-answering tasks. Answer the question based upon your knowledge. \n"
                      "Use three-to-five sentences maximum and keep the answer concise."),
}
CONTEXT_FORMAT = {  # how the context reaches the model, as shown in the cited cells (given)
    "simple_csv_rag 16": "{context} filled by create_stuff_documents_chain; cell 16 shows no per-block label",
    "local_rag_hf_faiss 22+24": "cell 24: f\"[Page {doc.metadata.get('page', '?')}]\\n{doc.page_content}\" joined by blank lines",
    "helper_functions L162-205": "{context} is whatever the caller passes; the function adds no label",
    "reliable_rag 18": "<doc{i+1}>:\\nTitle:{doc.metadata['title']}\\nSource:{doc.metadata['source']}\\nContent:{doc.page_content}",
    "captioning 20": "docs[0].page_content only (retriever k=1, cell 15); the 'name' metadata is not passed",
}
_F = ("context_only", "refusal", "cite_ids", "labelled_blocks", "invites_memory")
my_audit = {name: {f: None for f in _F} for name in PROMPTS}
# Fill it in, e.g. my_audit["simple_csv_rag 16"]["refusal"] = True

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p7_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
assert all(isinstance(v, bool) for row in my_audit.values() for v in row.values()), "every cell must be True or False"
assert _fp(my_audit) == "d75f2da525a7de19", "at least one judgement differs from the lesson's audit (S7). Re-read the exact words."
print("Task 6 OK")

**02 evidence: one failure the captioning prompt allows.** In 2–3 sentences, describe one concrete failure that `multi_model_rag_with_captioning.ipynb` cell 20 allows, and say whether it is a **retrieval** or a **generation** failure.

*Template:* "If the retrieved chunk ____, the prompt lets the model ____, and we cannot tell because ____. This is a ____ failure."

*Your answer:*

## Task 7 (optional, calls Groq): permissive vs grounded on an unanswerable question

**Not run by the course author. The expected behaviour is UNVERIFIED.** It sends question D-009 (the PDF names the IPCC but gives no founding year) with the same four chunks under two system prompts: the permissive `reliable_rag` prompt and your v0 prompt. It costs a few thousand Groq tokens.

Set `RUN_GROQ = True` to run it. It reads `.env` and never prints the key. Before running, write what you expect each prompt to do.

*My prediction:* permissive: ____ ; grounded v0: ____ .

In [ ]:
RUN_GROQ = False   # optional; set True to spend a few Groq tokens

if RUN_GROQ:
    import os
    from dotenv import load_dotenv, find_dotenv
    from groq import Groq
    load_dotenv(find_dotenv(usecwd=True))            # rag_learning/.env; the key is never printed
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    MODEL = os.environ["GROQ_MODEL_ID"]
    docs = "\n".join(f"<doc{i+1}>:\nContent:{c['text']}\n</doc{i+1}>\n" for i, c in enumerate(CHUNKS4))
    runs = {
        "permissive (reliable_rag 18)": (PROMPTS["reliable_rag 18"],
            f"Retrieved documents: \n\n <docs>{docs}</docs> \n\n User question: <question>{Q_D009}</question>"),
        "grounded (v0)": build_prompt(Q_D009, CHUNKS4),
    }
    for name, (s, u) in runs.items():
        r = client.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=1024, reasoning_effort="low",
            messages=[{"role": "system", "content": s}, {"role": "user", "content": u}])
        text = r.choices[0].message.content or ""
        assert text.strip(), f"empty answer (finish_reason={r.choices[0].finish_reason}): raise max_tokens"
        print(f"--- {name} | abstained={is_abstention(text)} | citations={extract_citations(text)} "
              f"| usage in/out={r.usage.prompt_tokens}/{r.usage.completion_tokens} | finish={r.choices[0].finish_reason}")
        print(text)
else:
    print("Task 7 skipped (RUN_GROQ = False).")

## Log it

Copy into your session log: your predictions next to the measured `tokens_k4`, `tokens_k8`, `tokens_2k`; your Task 1 and Task 4 sentences; your captioning-failure answer. If you ran Task 7, paste both answers and label them as **one run, n = 1, not a result**. The real comparison is Part I of the lesson, on the eval set.